# Agent 3 — Environment Agent (Package Edition)

This notebook is the **post-extraction demo** for Agent 3. All class and function definitions now live in the `immunosense.agents.environment` package. This notebook imports them and runs the same end-to-end validations.

**Architecture (unchanged):**
- Layer 1: regional/seasonal population baselines (EPA/NOAA)
- Layer 2: `process_environment_day(zip, date, source)` produces a daily summary
- Layer 3: `EnvironmentAgent` tracks personal baselines + runs BH FDR detection

**Validations performed:**
1. Live API smoke test (Charlotte, NC today) — real AirNow/OpenWeather/Google Pollen
2. PM2.5 trigger positive control — synthetic patient with planted trigger
3. UV trigger positive control — SLE-like photosensitive patient
4. Negative control — must produce zero patterns


In [ ]:
# === Package imports ===
import random as random_mod

import numpy as np
import pandas as pd

from immunosense.agents.environment import (
    EnvironmentAgent,
    process_environment_day,
    compute_flare_signature,
    DailyEnvironmentSummary,
    MockEnvironmentSource,
    CompositeEnvironmentSource,
)
from immunosense.agents.environment.sources import (
    AirNowSource, OpenWeatherSource, GooglePollenSource,
)

print('Imports OK')
print(f'EnvironmentAgent.agent_id     = {EnvironmentAgent.agent_id}')
print(f'EnvironmentAgent.output_dim   = {EnvironmentAgent.output_dim}')
print(f'EnvironmentAgent.agent_version = {EnvironmentAgent.agent_version}')


In [ ]:
# === Section 1: Live API smoke test (Charlotte, NC) ===
#
# Verifies the full Layer 1 + Layer 2 pipeline against real APIs.
# Falls back to mock gracefully if API keys are missing.
import datetime as _dt
_today = _dt.date.today().strftime('%Y-%m-%d')

summary = process_environment_day('28202', _today)

print(f'Date:     {summary.date}')
print(f'Location: ZIP {summary.location["zip"]} ({summary.location["region"]} / {summary.location["season"]})')
print(f'          lat={summary.location["lat"]:.3f}, lon={summary.location["lon"]:.3f}')
print()
print('Raw measurements:')
print(f'  pm25_ug_m3:            {summary.pm25_ug_m3}')
print(f'  ozone_ppb:             {summary.ozone_ppb}')
print(f'  uv_index:              {summary.uv_index}')
print(f'  barometric_change_kpa: {summary.barometric_change_kpa}')
print(f'  pollen_index:          {summary.pollen_index}')
print()
print('Population percentiles (vs regional/seasonal norms):')
for feature, pct in summary.percentiles.items():
    pct_str = f'{pct:.0%}' if pct is not None else 'N/A'
    print(f'  {feature:12s}: {pct_str}')
print()
print('EPA/WHO threshold alerts:')
for feature, alert in summary.threshold_alerts.items():
    print(f'  {feature:12s}: {alert}')
print()
print('Feature confidence:')
for feature, conf in summary.feature_confidence.items():
    src = summary.sources.get(feature, '?')
    print(f'  {feature:25s} {conf:10s}  (from {src})')
print()
print(f'Overall confidence: {summary.overall_confidence:.0%}  ({int(summary.overall_confidence*5)}/5 features real)')
print(f'Errors: {summary.errors if summary.errors else "none"}')


In [ ]:
# === Section 2: Layer 3 Validation - Synthetic Patient ===
#
# SyntheticEnvironmentPatient generates N days of trajectories with KNOWN triggers.
# Validates EnvironmentAgent end-to-end against three scenarios.
class SyntheticEnvironmentPatient:
    """Generates daily environmental records with calibrated triggers."""

    TRIGGER_THRESHOLDS = {
        'pm25_ug_m3':            15.0,   # vs N(10, 4)  -> ~10% of days
        'ozone_ppb':             55.0,   # vs N(45, 10) -> ~16% of days
        'uv_index':              7.0,    # vs N(5, 2)   -> ~16% of days
        'barometric_change_kpa': 0.7,    # vs |N(0, 0.4)| -> ~10% of days
        'pollen_index':          6.0,    # vs N(4, 2)   -> ~16% of days
    }

    BASELINE_DISTS = {
        'pm25_ug_m3':            (10.0, 4.0),
        'ozone_ppb':             (45.0, 10.0),
        'uv_index':              (5.0, 2.0),
        'barometric_change_kpa': (0.0, 0.4),   # abs() applied
        'pollen_index':          (4.0, 2.0),
    }

    def __init__(self, triggers=None, flare_lag_days=2,
                 trigger_flare_probability=0.85,
                 baseline_flare_probability=0.10, random_seed=42):
        self.triggers = triggers or []
        self.flare_lag_days = flare_lag_days
        self.trigger_flare_probability = trigger_flare_probability
        self.baseline_flare_probability = baseline_flare_probability
        self.rng = random_mod.Random(random_seed)
        self.np_rng = np.random.RandomState(random_seed)

    def _is_trigger_active(self, features):
        for trig in self.triggers:
            t, v = self.TRIGGER_THRESHOLDS.get(trig), features.get(trig)
            if t is not None and v is not None and v > t:
                return True
        return False

    def generate(self, n_days=60, start_date='2026-03-01'):
        records, scheduled_flares = [], {}
        start = pd.Timestamp(start_date)

        for i in range(n_days):
            date = (start + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
            features = {'date': date}
            for feat, (mean, std) in self.BASELINE_DISTS.items():
                v = self.np_rng.normal(mean, std)
                if feat == 'barometric_change_kpa':
                    v = abs(v)
                if feat == 'uv_index':
                    v = max(0.0, min(11.0, v))
                elif feat == 'pollen_index':
                    v = max(0.0, min(10.0, v))
                else:
                    v = max(0.0, v)
                features[feat] = v

            flare_prob = (self.trigger_flare_probability
                          if self._is_trigger_active(features)
                          else self.baseline_flare_probability)

            if self.rng.random() < flare_prob:
                flare_date = (start + pd.Timedelta(days=i + self.flare_lag_days)).strftime('%Y-%m-%d')
                severity = float(self.np_rng.uniform(1.5, 3.0))
                scheduled_flares[flare_date] = max(scheduled_flares.get(flare_date, 0.0), severity)

            records.append(features)

        return records, scheduled_flares


def _record_to_summary(record):
    """Convert SyntheticEnvironmentPatient record to a DailyEnvironmentSummary."""
    return DailyEnvironmentSummary(
        date=record['date'],
        location={'zip': '28202', 'lat': 35.227, 'lon': -80.843,
                   'region': 'SE', 'season': 'spring'},
        pm25_ug_m3=record['pm25_ug_m3'],
        ozone_ppb=record['ozone_ppb'],
        uv_index=record['uv_index'],
        barometric_change_kpa=record['barometric_change_kpa'],
        pollen_index=record['pollen_index'],
        overall_confidence=1.0,
    )


def run_validation_test(label, patient, expectation):
    records, flares = patient.generate(n_days=60)
    agent = EnvironmentAgent(n_permutations=500)
    for r in records:
        agent.observe(_record_to_summary(r))
    for d, s in flares.items():
        agent.observe_flare(d, s)
    report = agent.analyze()
    print(f'\n{label}')
    print(f'  Expectation: {expectation}')
    print(f'  Observed:     {report.n_days_observed} days, {report.n_flare_events} flare events')
    print(f'  Patterns ({len(report.detected_patterns)} found):')
    for p in report.detected_patterns:
        print(f'    {p.feature:30s} lag={p.lag_days}d  effect={p.effect_size:+.3f}'
              f'  p={p.p_value:.3f}  q={p.q_value:.3f}  [{p.confidence}]')


print('Synthetic patient + validation helper defined.')


In [ ]:
# === Section 3: Run the three validation tests ===
print('=' * 70)
print('Agent 3 Layer 3 validation - 3 tests')
print('=' * 70)

run_validation_test(
    'TEST 1: PM2.5 trigger (positive control)',
    SyntheticEnvironmentPatient(triggers=['pm25_ug_m3'], flare_lag_days=2, random_seed=42),
    'pm25_ug_m3 (>p85) at lag=2d (may be suppressed by BH if signal weak)',
)
run_validation_test(
    'TEST 2: UV trigger (SLE-like patient)',
    SyntheticEnvironmentPatient(triggers=['uv_index'], flare_lag_days=1, random_seed=43),
    'uv_index (>p85) at lag=1d',
)
run_validation_test(
    'TEST 3: Negative control (MUST be clean)',
    SyntheticEnvironmentPatient(triggers=[], random_seed=44),
    'Zero patterns at [high]; ideally zero patterns at all',
)

print('\n' + '=' * 70)
print('Validation complete.')
print('=' * 70)


In [ ]:
# === Section 4: Conductor-facing flare_signature demo ===
#
# The agent emits a 0-1 flare signature for any given day, used by the Conductor
# for cross-agent aggregation.
patient = SyntheticEnvironmentPatient(triggers=['pm25_ug_m3'], flare_lag_days=2, random_seed=42)
records, flares = patient.generate(n_days=60)
agent = EnvironmentAgent()
for r in records:
    agent.observe(_record_to_summary(r))
for d, s in flares.items():
    agent.observe_flare(d, s)

sig = agent.flare_signature()
print(f'Flare signature for most recent day:')
print(f'  Score: {sig["score"]:.3f}')
print(f'  Data quality confidence: {sig["data_quality_confidence"]:.3f}')
print(f'  Threshold breaches: {sig["threshold_breaches"] or "none"}')
print(f'\nTop 3 contributing factors:')
for c in sig['contributing_factors'][:3]:
    print(f'  {c["feature"]:25s} anomaly={c["anomaly_score"]}, weight={c["effective_weight"]:.3f}, trigger={c["is_personal_trigger"]}')
